# **RVC WebUI**

**Project cooked by [Phạm Huỳnh Anh](https://github.com/PhamHuynhAnh16)**


In [ ]:
#@title **⬇️ INSTALL**
import os
import shutil
import subprocess

from ipywidgets import Button
from IPython.display import clear_output, display

print("Installing...")

!rm -rf /tmp/RVC-clone > /dev/null 2>&1
!git clone https://github.com/Ezui0/RVC-HTML /tmp/RVC-clone > /dev/null 2>&1

if os.path.isdir("/tmp/RVC-clone"):
    for item in os.listdir("/tmp/RVC-clone"):
        src = os.path.join("/tmp/RVC-clone", item)
        dst = os.path.join("/content", item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)

    !rm -rf /tmp/RVC-clone

    os.makedirs("rvc_models", exist_ok=True)

    print("Installing dependencies...")
    ok = subprocess.call("if command -v uv > /dev/null 2>&1; then uv pip install --system -q -r /content/requirements.txt; else pip install -q -r /content/requirements.txt; fi", shell=True) == 0

    if not ok:
        print("[WARNING] uv failed, retrying with pip...")
        ok = subprocess.call("pip install -q -r /content/requirements.txt", shell=True) == 0

    if ok:
        clear_output()
        display(Button(description="\u2714 Success", button_style="success"))
    else:
        print("[ERROR] Dependency installation failed. Read the error above, then run this cell again.")
        display(Button(description="\u2716 Install Failed", button_style="danger"))
else:
    clear_output()
    print("[ERROR] Failed to clone the repository. Check your internet connection, then run this cell again.")
    display(Button(description="\u2716 Clone Failed", button_style="danger"))


In [ ]:
#@title **📩 Download model**
from ipywidgets import Button
from google.colab import files
from IPython.display import clear_output, display

#@markdown **Support urls from huggingface.co / drive.google.com / mega.nz / mediafire.com**

modelname = "" # @param {"type":"string","placeholder":"Model Name"}
url = "" # @param {"type":"string","placeholder":"https://huggingface.co//..."}

success = False

if not url:
    uploaded = files.upload()

    if not uploaded:
        print("[WARNING] No file uploaded.")
    else:
        args = f'from modules.download import save_drop_model; save_drop_model(\\"{list(uploaded.keys())[0]}\\")'
        !python3 -c "$args"
        success = True
else:
    args = f'from modules.download import download_model; download_model(\\"{url}\\", \\"{modelname}\\")'
    !python3 -c "$args"
    success = True

clear_output()
if success:
    display(Button(description="\u2714 Success", button_style="success"))
else:
    display(Button(description="\u26a0 No file uploaded", button_style="warning"))


In [ ]:
#@title **🚀 Start WebUI (app.py)**
import os
import re
import time
import subprocess

from IPython.display import HTML, display

print("Starting RVC WebUI server...")

# stop previous instance (cell re-run), then start the server in background
subprocess.run("pkill -f 'python3 app.py'", shell=True)
time.sleep(1)
server_log = open("server.log", "w")
process = subprocess.Popen(
    ["python3", "app.py"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

public_url = ""
for _ in range(90):
    if process.poll() is not None:
        break  # server exited early
    time.sleep(1)
    try:
        log = open("server.log").read()
    except OSError:
        log = ""
    match = re.search(r"https://[\w-]+\.gradio\.live", log)
    if match:
        public_url = match.group(0)
        break

if process.poll() is not None:
    print("[ERROR] Server exited unexpectedly. server.log:")
    try:
        print(open("server.log").read()[-4000:])
    except OSError:
        print("(no log)")
else:
    # fallback: Colab built-in proxy (works even if the gradio share tunnel fails)
    if not public_url:
        try:
            from google.colab.output import eval_js
            public_url = eval_js("google.colab.kernel.proxyPort(7860)")
        except Exception:
            pass

    if public_url:
        print(f"[INFO] WebUI is ready: {public_url}")
        print("[INFO] The server keeps running while this Colab session is alive.")
        display(HTML(f'<a href="{public_url}" target="_blank"><b>Open RVC WebUI: {public_url}</b></a>'))
    else:
        print("[ERROR] No public URL found. server.log:")
        try:
            print(open("server.log").read()[-4000:])
        except OSError:
            print("(no log)")
